# Multi-attribute BRB systems

This notebook demonstrates BRB systems with multiple input attributes, covering two examples:

1. **$f(x_1, x_2) = x_1 + x_2$** — a simple additive function from Misitano (2020), Section 3.5 Example 1
2. **Himmelblau's function** — a challenging nonlinear surface from Chen et al. (2011), Section 5.1, Eq. 9

```
# requires: pip install matplotlib
```

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from desdeo_brb import BRBModel
from desdeo_brb.utils import build_rule_antecedent_indices

## Example 1: $f(x_1, x_2) = x_1 + x_2$

With two attributes each having 3 referential values, the Cartesian product yields $3 \times 3 = 9$ rules. The rule base construction follows Eq. 3.22–3.24 in the thesis (Misitano, 2020).

In [ ]:
prv = [np.array([0.0, 1.0, 2.0]), np.array([0.0, 1.0, 2.0])]
crv = np.array([0.0, 1.0, 2.0, 3.0, 4.0])

model = BRBModel(prv, crv, initial_rule_fn=lambda x: x[0] + x[1])

print(f"Rules: {model.rule_base.n_rules} (3 x 3 Cartesian product)")
print(f"Attributes: {model.rule_base.n_attributes}")
print(f"Consequents: {model.rule_base.n_consequents}")

In [ ]:
# Show the 9 rules
rb = model.rule_base
indices = rb.rule_antecedent_indices
for k in range(rb.n_rules):
    x1 = prv[0][indices[k, 0]]
    x2 = prv[1][indices[k, 1]]
    y = x1 + x2
    print(
        f"Rule {k + 1}: x1={x1:.0f}, x2={x2:.0f} -> f={y:.0f}  beliefs={np.round(rb.belief_degrees[k], 2)}"
    )

In [ ]:
# Verify exact match at referential value combinations
indices = build_rule_antecedent_indices(prv)
X_ref = np.column_stack([prv[0][indices[:, 0]], prv[1][indices[:, 1]]])
y_true = X_ref[:, 0] + X_ref[:, 1]
y_pred = model.predict_values(X_ref)

print("Prediction accuracy at referential values:")
for i in range(len(X_ref)):
    print(
        f"  ({X_ref[i, 0]:.0f}, {X_ref[i, 1]:.0f}): true={y_true[i]:.1f}, pred={y_pred[i]:.4f}"
    )

In [ ]:
# Predict on a grid and plot
x1 = np.linspace(0, 2, 50)
x2 = np.linspace(0, 2, 50)
X1, X2 = np.meshgrid(x1, x2)
X_grid = np.column_stack([X1.ravel(), X2.ravel()])
Y_pred = model.predict_values(X_grid).reshape(50, 50)
Y_true = X1 + X2

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
c1 = axes[0].contourf(X1, X2, Y_true, levels=20, cmap="viridis")
axes[0].set_title("True $f(x_1, x_2) = x_1 + x_2$")
axes[0].set_xlabel("$x_1$")
axes[0].set_ylabel("$x_2$")
plt.colorbar(c1, ax=axes[0])

c2 = axes[1].contourf(X1, X2, Y_pred, levels=20, cmap="viridis")
axes[1].set_title("BRB prediction (untrained)")
axes[1].set_xlabel("$x_1$")
axes[1].set_ylabel("$x_2$")
plt.colorbar(c2, ax=axes[1])

plt.tight_layout()
plt.show()

## Example 2: Himmelblau's function

Himmelblau's function $f(x, y) = (x^2 + y - 11)^2 + (x + y^2 - 7)^2$ is a standard benchmark with four local minima. This example follows **Chen et al. (2011), Section 5.1, Eq. 9**, using 7 referential values per attribute on $[-6, 6]^2$, yielding $7 \times 7 = 49$ rules.

In [ ]:
def himmelblau(x):
    return (x[0] ** 2 + x[1] - 11) ** 2 + (x[0] + x[1] ** 2 - 7) ** 2


rv_1d = np.array([-6.0, -4.0, -2.0, 0.0, 2.0, 4.0, 6.0])
prv_h = [rv_1d, rv_1d]
crv_h = np.array([0.0, 200.0, 500.0, 1000.0, 2200.0])

model_h = BRBModel(prv_h, crv_h, initial_rule_fn=himmelblau)
print(f"Rules: {model_h.rule_base.n_rules}")

In [ ]:
# Training grid (13 x 13 = 169 points)
x_train = np.linspace(-6, 6, 13)
X1t, X2t = np.meshgrid(x_train, x_train, indexing="ij")
X_train = np.column_stack([X1t.ravel(), X2t.ravel()])
y_train = np.array([himmelblau(row) for row in X_train])

# Evaluate before training
x_eval = np.linspace(-6, 6, 40)
X1e, X2e = np.meshgrid(x_eval, x_eval)
X_eval = np.column_stack([X1e.ravel(), X2e.ravel()])
y_eval_true = np.array([himmelblau(row) for row in X_eval])

y_before = model_h.predict_values(X_eval)
mse_before = np.mean((y_eval_true - y_before) ** 2)

# Train
model_h.fit(X_train, y_train, fix_endpoints=True)

y_after = model_h.predict_values(X_eval)
mse_after = np.mean((y_eval_true - y_after) ** 2)

print(f"MSE before training: {mse_before:.0f}")
print(f"MSE after training:  {mse_after:.0f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

levels = np.linspace(0, 2200, 20)

c0 = axes[0].contourf(
    X1e, X2e, y_eval_true.reshape(40, 40), levels=levels, cmap="plasma"
)
axes[0].set_title("True Himmelblau")
axes[0].set_xlabel("$x_1$")
axes[0].set_ylabel("$x_2$")

c1 = axes[1].contourf(X1e, X2e, y_before.reshape(40, 40), levels=levels, cmap="plasma")
axes[1].set_title("BRB (untrained)")
axes[1].set_xlabel("$x_1$")

c2 = axes[2].contourf(X1e, X2e, y_after.reshape(40, 40), levels=levels, cmap="plasma")
axes[2].set_title("BRB (trained)")
axes[2].set_xlabel("$x_1$")

plt.colorbar(c0, ax=axes, shrink=0.8)
plt.show()

## Referential value placement

The accuracy of a BRB model depends heavily on where the referential values are placed. In the uniform grid above, some regions of the Himmelblau surface are better approximated than others.

The `fit()` method can optimize the referential value positions themselves (with `fix_endpoints=True` keeping the domain boundaries fixed). This **adaptive referential value training** is described in Chen et al. (2011), Section 4.2, and can significantly improve accuracy without adding more rules.

You can inspect the trained referential values to see how the optimizer shifted them:

In [ ]:
print("Original referential values:", rv_1d)
print("Trained (attr 0):", np.round(model_h.precedent_referential_values[0], 2))
print("Trained (attr 1):", np.round(model_h.precedent_referential_values[1], 2))